<style>
.reveal { font-family: 'Segoe UI', system-ui, sans-serif; font-size: 1.05em; }
.reveal h2 { color: #0D2240; border-bottom: 2.5px solid #1A7A9A; padding-bottom: .3em; }
.reveal h3 { color: #1A7A9A; }
.reveal .slides section { text-align: left; }
.reveal pre { font-size: .75em; box-shadow: none; border-left: 3px solid #1A7A9A; }
.defn { background:#EAF6FA; border-left:4px solid #1A7A9A; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.nota { background:#FFF8E1; border-left:4px solid #C8961E; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
.alerta { background:#FDE8E8; border-left:4px solid #C0392B; padding:.6em 1em; margin:.5em 0; border-radius:4px; }
</style>

## CMTC: Estado Estable y Colas M/M/1
### T2.3 (parte b) · Modelado de Sistemas bajo Incertidumbre
Universidad de los Andes · Ingeniería Industrial

## Agenda
1. Distribución estacionaria: existencia y unicidad
2. Ecuaciones de balance global
3. Balance detallado y reversibilidad
4. Cadenas de nacimiento-muerte
5. Cola M/M/1: derivación y fórmulas de Erlang
6. Sensibilidad a ρ
7. Caso: Metro de Bogotá

## Distribución estacionaria
<div class='defn'>
El vector π∞ = [π∞₀, π∞₁, ..., π∞ₙ₋₁] es la <strong>distribución estacionaria</strong> si:

$$\pi^\infty Q = \mathbf{0}, \qquad \sum_i \pi^\infty_i = 1$$

Bajo condiciones de irreducibilidad y aperiodicidad, π(t) → π∞ cuando t → ∞, independientemente de π(0).
</div>

## Ecuaciones de balance global
La condición πQ = 0 equivale a:

$$\forall j: \quad \pi_j^\infty \, q_j = \sum_{i \neq j} \pi_i^\infty \, q_{ij}$$

**Interpretación (Ley de flujo):**
En estado estacionario, el flujo de probabilidad **saliente** del estado j es igual al flujo **entrante** desde todos los demás estados.

<div class='nota'>
El sistema πQ = 0, ∑πᵢ = 1 tiene siempre una única solución para CMTCs irreducibles de espacio finito. Se resuelve con álgebra lineal reemplazando una ecuación de balance por la restricción de normalización.
</div>

## Balance detallado (reversibilidad)
<div class='defn'>
Una CMTC satisface <strong>balance detallado</strong> si para todo par (i, j):

$$\pi_i^\infty \, q_{ij} = \pi_j^\infty \, q_{ji}$$

El flujo de i a j en estado estacionario iguala el flujo de j a i.
</div>

El balance detallado implica balance global, pero no viceversa.

Las cadenas de **nacimiento-muerte** siempre satisfacen balance detallado (estructura tridiagonal).

## Cadenas de nacimiento-muerte
**Espacio de estados:** {0, 1, 2, ...}
Solo se permiten transiciones entre estados adyacentes:
- Tasa de nacimiento (subida): λᵢ = tasa de i → i+1
- Tasa de muerte (bajada): μᵢ = tasa de i → i-1

**Balance detallado:** π∞ᵢ λᵢ = π∞ᵢ₊₁ μᵢ₊₁

**Solución recursiva:**
$$\pi_n^\infty = \pi_0^\infty \prod_{k=0}^{n-1} \frac{\lambda_k}{\mu_{k+1}}$$

con π∞₀ determinada por la normalización.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Cola M/M/1: lambda_i = lambda, mu_i = mu para todo i > 0
lam, mu = 3.0, 5.0
rho = lam / mu
N_max = 20   # truncar en N_max estados

# Distribución estacionaria exacta
pi = np.zeros(N_max + 1)
pi[0] = 1 - rho                         # formula exacta M/M/1
for n in range(1, N_max + 1):
    pi[n] = (1 - rho) * rho**n

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(N_max + 1), pi, color='#1A7A9A', edgecolor='white', alpha=0.85)
ax.set_xlabel('Número de clientes en el sistema (n)')
ax.set_ylabel('π∞ₙ')
ax.set_title(f'Distribución estacionaria M/M/1  (λ={lam}, μ={mu}, ρ={rho:.2f})')
# Métricas analíticas
L = rho / (1 - rho)          # E[número en sistema]
Lq = rho**2 / (1 - rho)      # E[número en cola]
W = 1 / (mu - lam)           # E[tiempo en sistema]
Wq = rho / (mu - lam)        # E[tiempo en cola]
ax.text(0.65, 0.75,
        f"L = {L:.3f}
Lq = {Lq:.3f}
W = {W:.3f}
Wq = {Wq:.3f}",
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round', fc='#EAF6FA', ec='#1A7A9A'))
plt.tight_layout(); plt.show()

## Fórmulas de Erlang — Cola M/M/1
Para λ < μ (condición de estabilidad ρ < 1):

| Métrica | Fórmula | Descripción |
|---|---|---|
| **ρ** | λ/μ | Utilización del servidor |
| **L** | ρ/(1−ρ) | Clientes promedio en el sistema |
| **Lq** | ρ²/(1−ρ) | Clientes promedio en la cola |
| **W** | 1/(μ−λ) | Tiempo promedio en el sistema |
| **Wq** | ρ/(μ−λ) | Tiempo promedio en la cola |
| **P(n)** | (1−ρ)ρⁿ | P(exactamente n clientes) |

**Ley de Little:** L = λW, Lq = λWq

In [ ]:
# Sensibilidad a ρ
rhos = np.linspace(0.01, 0.99, 300)
L_vals  = rhos / (1 - rhos)
Lq_vals = rhos**2 / (1 - rhos)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(rhos, L_vals,  color='#1A7A9A', lw=2.5, label='L  (en sistema)')
axes[0].plot(rhos, Lq_vals, color='#C8961E', lw=2.5, label='Lq (en cola)')
axes[0].axvline(0.9, ls='--', color='#C0392B', lw=1.5, label='ρ = 0.9')
axes[0].set_xlabel('Utilización ρ = λ/μ')
axes[0].set_ylabel('Número promedio de clientes')
axes[0].set_title('M/M/1: L y Lq en función de ρ')
axes[0].set_ylim(0, 20); axes[0].legend()

# W como función de ρ (fijando μ = 1)
mu_fixed = 1.0
W_vals = 1 / (mu_fixed - rhos * mu_fixed)
axes[1].plot(rhos, W_vals, color='#1A7A9A', lw=2.5)
axes[1].axvline(0.9, ls='--', color='#C0392B', lw=1.5, label='ρ = 0.9')
axes[1].set_xlabel('Utilización ρ')
axes[1].set_ylabel('W (tiempo promedio en sistema)')
axes[1].set_title('M/M/1: W en función de ρ  (μ=1)')
axes[1].set_ylim(0, 20); axes[1].legend()
plt.tight_layout(); plt.show()

## Caso: Metro de Bogotá
**Situación:** Pasajeros llegan a una estación de Transmilenio/Metro siguiendo un PP(λ).
El sistema de torniquetes procesa a tasa μ (1 servidor efectivo en hora pico).

**Datos observados (hora pico):**
- Llegadas: ~360 personas/hora → λ = 6 por minuto
- Tiempo de validación promedio: 12 segundos → μ = 5 por minuto

**Pregunta:** ¿Es estable el sistema? ¿Cuánto espera un pasajero en promedio?

In [ ]:
# Metro de Bogotá — modelo M/M/1
lam_metro = 6.0    # pasajeros/minuto
mu_metro  = 5.0    # validaciones/minuto (capacidad del torniquete)
rho_metro = lam_metro / mu_metro

print(f"Utilización ρ = {rho_metro:.2f}")
if rho_metro >= 1:
    print("SISTEMA INESTABLE: la cola crece indefinidamente.")
    print(f"
Se requiere aumentar la capacidad:")
    mu_min = lam_metro * 1.1   # margen 10%
    print(f"  μ mínimo recomendado = {mu_min:.1f} validaciones/min")
    print(f"  = {60/mu_min:.1f} segundos por validación")
    print(f"  Alternativa: agregar {int(np.ceil(rho_metro))} torniquetes")
else:
    L  = rho_metro / (1 - rho_metro)
    W  = 1 / (mu_metro - lam_metro)
    Wq = rho_metro / (mu_metro - lam_metro)
    print(f"L  = {L:.2f} pasajeros en sistema")
    print(f"W  = {W:.2f} min en sistema")
    print(f"Wq = {Wq:.2f} min en cola")

## Conclusiones
- La distribución estacionaria π∞ satisface π∞Q = 0, ∑π∞ᵢ = 1.
- El balance detallado (queueing reversibility) simplifica la solución en cadenas B-M.
- La cola M/M/1 tiene solución geométrica exacta con parámetro ρ = λ/μ.
- El sistema es **solo estable si ρ < 1**; para ρ cercano a 1 los tiempos de espera crecen drásticamente.
- La Ley de Little (L = λW) conecta número en sistema con tiempo de permanencia.

**Módulo 3:** Modelado de entradas, V&V y análisis estadístico de salida de simulación.